# Pydantic Computed Fields: Serializing Properties

In standard object-oriented design, dynamic data is often exposed using properties. However, standard Python properties are not automatically included when serialization functions (like converting a model to a dictionary or a JSON string) are run.

Pydantic provides `@computed_field` to resolve this. By decorating a `@property` with `@computed_field`, you instruct Pydantic to treat it as a model field, meaning it will be automatically calculated and included in all serialized outputs.

In this notebook, we cover:
1. Adding dynamic read-only properties using Python's `@property`.
2. Exposing properties in serialization using Pydantic's `@computed_field`.
3. Calculating a patient's BMI dynamically based on weight and height.


In [1]:
!uv pip install pydantic 'pydantic[email]' --quiet


## 1. Imports and Sub-Models


In [2]:
from pydantic import BaseModel, EmailStr, AnyUrl, Field, field_validator, model_validator, computed_field
from typing import List, Dict, Optional, Annotated


In [3]:
class ContactDetails(BaseModel):
    email_id: Annotated[EmailStr, Field(description='Primary email address of the patient', examples=['john@example.com'])]
    contact_number: Annotated[str, Field(min_length=10, max_length=15, description='Primary contact number', examples=['9999999999'])]
    emergency_contact_number: Annotated[Optional[str], Field(default=None, min_length=10, max_length=15, description='Emergency contact number', examples=['8888888888'])]

    @field_validator('email_id')
    @classmethod
    def email_validator(cls, value: str) -> str:
        valid_domain = ['domain.io', 'example.com']
        domain_name = value.split("@")[-1]

        if domain_name not in valid_domain:
            raise ValueError('Not a valid domain')
        
        return value


## 2. Defining Computed Fields on Patient

We declare a `height` field and a dynamic property `calculate_bmi`.
Applying `@computed_field` tells Pydantic to include `calculate_bmi` in print statements, `model_dump()`, and `model_dump_json()`.


In [4]:
class Patient(BaseModel):

    name: str = Annotated[str, Field(max_length=150, title='Name of the patient', description='Patient Name for records', examples=['John Doe'])]
    age: int
    linkedin_url: Optional[AnyUrl] = None
    weight: Annotated[float, Field(gt=0, description='Submit patient weight for the report', strict=True)]
    
    # Adding height for BMI calculation
    height: Annotated[float, Field(gt=0, description='Submit patient height for the report', strict=True)]
    married: Annotated[bool, Field(default=None, description='Is the patient married or not')]
    allergies: Annotated[Optional[List[str]], Field(default=None, max_length=5)]
    contact_details: ContactDetails

    @field_validator('name')
    @classmethod
    def transform_name(cls, value: str) -> str:
        return value.upper()
    
    @field_validator('age', mode='before')
    @classmethod
    def validate_age(cls, value: int) -> int:
        if 0 < value < 120:
            return value
        else:
            raise ValueError("Age should be in between 0 and 120")

    @model_validator(mode='after')
    def validate_emergency_contact(self):
        if self.age > 60 and self.contact_details.emergency_contact_number is None:
            raise ValueError('Emergency contact number is mandatory for patients above 60 years of age')
        return self
    
    # Declares a computed field that gets serialized
    @computed_field
    @property
    def calculate_bmi(self) -> float:
        # BMI formula: weight (kg) / height^2 (m^2)
        bmi = round(self.weight / (self.height**2), 2)
        return bmi


## 3. Data Pipelines & Simulation


In [5]:
def insert_patient_data(patient: Patient):
    print(f"Patient Name: {patient.name}")
    print(f"Patient Age: {patient.age}")
    print(f"Patient Weight: {patient.weight}")
    print(f"Patient Married Status: {patient.married}")
    print(f"Patient Allergies: {patient.allergies}")
    print(f"Patient Contact Details: {patient.contact_details}")
    
    # We can access computed fields as regular properties
    print(f"Patient Calculated BMI: {patient.calculate_bmi}")
    print('Patient info inserted')


In [6]:
def update_patient_data(patient: Patient):
    print(f"Patient Name: {patient.name}")
    print(f"Patient Age: {patient.age}")
    print(f"Patient Weight: {patient.weight}")
    print(f"Patient Married Status: {patient.married}")
    print(f"Patient Allergies: {patient.allergies}")
    print(f"Patient Contact Details: {patient.contact_details}")
    print(f"Patient Calculated BMI: {patient.calculate_bmi}")
    print('Patient info updateed')


In [7]:
patient_info = {
    'name': 'Kevin',
    'age': 26,
    'weight': 67.9,
    'height': 1.73,
    'married': False,
    'allergies': ['lactose', 'dust'],
    'contact_details': {
        'email_id': 'example@domain.io',
        'contact_number': '9999999999'
    }
}


## 4. Executing and Inspecting Serialized Output

When printing `patient`, note that `calculate_bmi=22.69` appears automatically inside the model's print representation!


In [8]:
patient = Patient(**patient_info)
patient


Patient(name='KEVIN', age=26, linkedin_url=None, weight=67.9, height=1.73, married=False, allergies=['lactose', 'dust'], contact_details=ContactDetails(email_id='example@domain.io', contact_number='9999999999', emergency_contact_number=None), calculate_bmi=22.69)

In [9]:
insert_patient_data(patient=patient)


Patient Name: KEVIN
Patient Age: 26
Patient Weight: 67.9
Patient Married Status: False
Patient Allergies: ['lactose', 'dust']
Patient Contact Details: email_id='example@domain.io' contact_number='9999999999' emergency_contact_number=None
Patient Calculated BMI: 22.69
Patient info inserted


In [10]:
update_patient_data(patient=patient)


Patient Name: KEVIN
Patient Age: 26
Patient Weight: 67.9
Patient Married Status: False
Patient Allergies: ['lactose', 'dust']
Patient Contact Details: email_id='example@domain.io' contact_number='9999999999' emergency_contact_number=None
Patient Calculated BMI: 22.69
Patient info updateed
